# TFP3 Technology Benchmark — Initial Comparison

_2026-04-29 — Adam Klie_

Consolidated walk-through of what the cross-technology benchmark sweep tells us so far. Source data lives in `results/cross_tech_comparison/` (per-axis subdirs: `cross_parameter/`, `cells_vs_expected/`, `per_dataset_summary/`, `scrublet_compare/`); narrative lives in `scratch/2026_04_28_manuscript_brainstorm/benchmark_sweep_status.md`. This notebook pulls the numbers and plots together for sharing with the GPS team.

**Datasets compared:** Hon (UTSW), Huangfu (MSKCC), Gersbach GEM-Xv3 (Duke), Gersbach HTv2 (Duke), Engreitz CC-Perturb-seq (Stanford). All on the same WTC11 → iPSC system with the same dCas9-KRAB CLYBL-locus CRISPRi construct and a shared 416-guide library; the only experimental axis that varies is the scRNA chemistry / sequencing technology.

**Pipeline configs swept:** 200 / 500 / 800 / 2000 UMI floors, knee2 automatic, plus a clean scrublet on/off pair (verified from `pipeline_info/params_*.json` — both use sceptre, only `ENABLE_SCRUBLET` differs).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['figure.dpi'] = 110

PROJECT_ROOT = Path('/cellar/users/aklie/projects/tf_perturb_seq')
BASE = PROJECT_ROOT / 'datasets' / 'technology-benchmark_WTC11_TF-Perturb-seq'
RESULTS = BASE / 'results' / 'cross_tech_comparison'
OUT = BASE / 'results' / 'initial_comparison'
OUT.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT / 'config'))
from loader import load_colors
cfg = load_colors('technology-benchmark_WTC11_TF-Perturb-seq')
DATASET_COLORS = cfg['dataset_colors']
DATASET_ORDER = cfg['dataset_order']

SHORT = {
    'Hon_WTC11-benchmark_TF-Perturb-seq': 'Hon',
    'Huangfu_WTC11-benchmark_TF-Perturb-seq': 'Huangfu',
    'Gersbach_WTC11-benchmark_TF-Perturb-seq_GEM-Xv3': 'Gersbach GEM-Xv3',
    'Gersbach_WTC11-benchmark_TF-Perturb-seq_HTv2': 'Gersbach HTv2',
    'Engreitz_WTC11-benchmark_TF-Perturb-seq': 'Engreitz',
}

FILTER_CONFIG_ORDER = ['200 UMI', '500 UMI', '800 UMI', '2000 UMI', 'Knee2']

def fmt_k(x, _):
    return f'{x/1000:.0f}K'

def short_axis(ds_list):
    return [SHORT.get(d, d).replace(' ', '\n', 1) for d in ds_list]

print(f'Output dir: {OUT}')

## 1. Datasets and library design

Five labs ran the same TFP3 guide library through different scRNA-seq chemistries on a shared WTC11→iPSC system.

In [ ]:
overview = pd.read_csv(BASE / 'dataset_overview.tsv', sep='\t')
overview['short_name'] = overview['dataset'].map(SHORT)
display_cols = ['short_name', 'lab', 'institution', 'sequencing_technology', 'expected_cells', 'contact']
display(overview[display_cols].set_index('short_name'))

# Library composition (5 classes)
lib = pd.DataFrame([
    {'class': 'non_targeting', 'count': 30, 'description': 'No genomic target (NTC)'},
    {'class': 'negative_control_OR', 'count': 54, 'description': '9 olfactory receptor genes × 6 guides each'},
    {'class': 'positive_control', 'count': 8, 'description': 'CD81, CD55, CD151, NGFRAP1, TFRC'},
    {'class': 'tf_targeting', 'count': 324, 'description': 'TFs of interest (~50 genes, multiple guides each)'},
])
lib.loc[len(lib)] = ['TOTAL', 416, '']
print('\nLibrary composition (identical across all 5 datasets, 416 guides):')
display(lib.set_index('class'))

## 2. Cell yield across filter configs vs experimental design

How many cells survive each cell-calling threshold, compared to what each lab expected.

In [ ]:
cve = pd.read_csv(RESULTS / 'cells_vs_expected' / 'cells_vs_expected.tsv', sep='\t')
ds_in = [d for d in DATASET_ORDER if d in cve['dataset'].values]

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5), sharey=False)
for ax, ds in zip(axes, ds_in):
    sub = cve[cve['dataset'] == ds].set_index('run_label').reindex(FILTER_CONFIG_ORDER)
    color = DATASET_COLORS[ds]
    x = np.arange(len(FILTER_CONFIG_ORDER))
    ax.bar(x, sub['n_cells'].values, color=color, alpha=0.85, edgecolor='white')
    exp = sub['expected_cells'].dropna().iloc[0] if sub['expected_cells'].notna().any() else None
    if exp:
        ax.axhline(exp, color='black', linestyle='--', linewidth=1.5)
        ax.text(0, exp*1.05, f'expected {exp/1000:.0f}K', fontsize=8)
    for xi, v in zip(x, sub['n_cells'].values):
        if pd.notna(v) and exp:
            ax.text(xi, v*1.02, f'{v/exp:.1f}×', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(FILTER_CONFIG_ORDER, rotation=30, ha='right', fontsize=9)
    ax.set_title(SHORT.get(ds, ds), fontsize=11, color=color, fontweight='bold')
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_k))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
axes[0].set_ylabel('n_cells')
fig.suptitle('Cells per filter config vs experimental design (dashed = expected)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT / '02_cells_vs_expected.pdf', bbox_inches='tight')
plt.show()

**What stands out:**
- **Hon: severely under-expected** at every filter (~30% of expected 210K cells). Likely HTO demultiplexing overhead — the only dataset using hashtag oligos.
- **Gersbach pair: explodes at 200 UMI** — GEM-Xv3 hits 6.4× expected (513K vs 80K), HTv2 hits 3.3× (263K vs 80K). 200 UMI floor admits many empty / non-cell barcodes for these techs.
- **Engreitz at Knee2: 10% of expected** — 20K vs 200K. This is the suspected lane-splitting artifact, not biology (caveat baked into `plot_params_sweep.py`).
- **Huangfu at 200 UMI: 1.17×** — the only dataset that approximately matches its experimental design at the most permissive filter.

The right cell-calling floor is dataset-dependent. There's no single "best" config across all 5.

## 3. Knockdown sensitivity (auROC / auPRC) across filter configs

In [ ]:
wide = pd.read_csv(RESULTS / 'per_dataset_summary' / 'summary_wide.tsv', sep='\t')
ds_in = [d for d in DATASET_ORDER if d in wide['dataset'].values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ['auroc', 'auprc'], ['auROC', 'auPRC']):
    for ds in ds_in:
        sub = wide[wide['dataset'] == ds].set_index('config_label').reindex(FILTER_CONFIG_ORDER)
        ax.plot(FILTER_CONFIG_ORDER, sub[metric].values, 'o-', color=DATASET_COLORS[ds],
                label=SHORT.get(ds, ds), linewidth=2, markersize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0.5, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', rotation=20)
axes[0].legend(loc='lower right', fontsize=8)
axes[0].set_ylabel('auROC')
axes[1].set_ylabel('auPRC')
fig.suptitle('Knockdown sensitivity across filter configs (intended-target evaluation)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT / '03_auroc_auprc_across_configs.pdf', bbox_inches='tight')
plt.show()

# Best AUROC config per dataset
best = pd.read_csv(RESULTS / 'per_dataset_summary' / 'summary_best.tsv', sep='\t')
best['short_name'] = best['dataset'].map(SHORT)
print('\nBest auROC config per dataset:')
display(best[['short_name', 'config_label', 'auroc', 'auprc', 'n_cells', 'ratio_obs_to_expected', 'frac_cells_with_guide']].set_index('short_name').round(3))

**What stands out:**
- **AUROC-best config differs by dataset**: Hon→500, Huangfu→Knee2, Gersbach pair→200, Engreitz→500.
- **Range of AUROC across configs varies hugely**: tight for Hon (0.04) and Huangfu (0.02); wide for Gersbach HTv2 (0.11) and Engreitz (0.18). **Engreitz is the most filter-sensitive dataset.**
- ⚠ **AUROC is not a sufficient ranking metric** — for the Gersbach pair, "best AUROC" 200 UMI gives 3-6× the expected cell count with only 17-33% of those cells having a guide assigned. The high AUROC is on a tiny effective subset. Need to balance against `frac_cells_with_guide` × cell yield.

## 4. Per-class fold change at each dataset's best config

Two flavors:
- **Intended-target log2FC** — FC on the specific intended target gene. Available for positive controls + TF-targeting (NT excluded — no intended target).
- **Trans median log2FC per guide** — median log2FC across all genes tested per guide. Works for all 4 classes including non-targeting and OR negative controls.

In [ ]:
# Intended-target log2FC by class (best config per dataset)
best['short_name'] = best['dataset'].map(SHORT)
intended_cols = ['median_log2fc_positive_control', 'median_log2fc_tf_targeting',
                 'median_log2fc_negative_control_OR', 'median_log2fc_non_targeting']
trans_cols = ['trans_median_log2fc_positive_control', 'trans_median_log2fc_tf_targeting',
              'trans_median_log2fc_negative_control_OR', 'trans_median_log2fc_non_targeting']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ds_in = [d for d in DATASET_ORDER if d in best['dataset'].values]
x = np.arange(len(ds_in))
width = 0.2
class_labels = ['Pos. control', 'TF-targeting', 'OR neg ctrl', 'Non-targeting']
class_colors = ['#d62728', '#1f77b4', '#9467bd', '#7f7f7f']

# Panel 1: intended-target log2fc
for j, (col, label, c) in enumerate(zip(intended_cols, class_labels, class_colors)):
    vals = [best[best['dataset'] == d][col].iloc[0] for d in ds_in]
    offset = (j - 1.5) * width
    axes[0].bar(x + offset, vals, width, color=c, alpha=0.85, label=label)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels([SHORT.get(d, d) for d in ds_in], fontsize=9, rotation=20, ha='right')
axes[0].set_title('Intended-target log2FC (NT excluded)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('median log2FC')
axes[0].legend(fontsize=8, loc='lower left')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Panel 2: trans median log2fc per guide
for j, (col, label, c) in enumerate(zip(trans_cols, class_labels, class_colors)):
    vals = [best[best['dataset'] == d][col].iloc[0] for d in ds_in]
    offset = (j - 1.5) * width
    axes[1].bar(x + offset, vals, width, color=c, alpha=0.85, label=label)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels([SHORT.get(d, d) for d in ds_in], fontsize=9, rotation=20, ha='right')
axes[1].set_title('Trans median log2FC per guide (all 4 classes)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('median log2FC')
axes[1].legend(fontsize=8, loc='lower left')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

fig.suptitle("Per-class fold change at each dataset's best AUROC config", fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT / '04_per_class_log2fc.pdf', bbox_inches='tight')
plt.show()

**What stands out:**
- **Positive controls are robustly knocked down** across all datasets (intended log2FC -1.6 to -2.4).
- **TF-targeting log2FC** ranges -0.34 (Gersbach HTv2) to -0.67 (Huangfu) — Huangfu is the most sensitive technology for the average TF.
- **NT and OR null classes track each other closely** in every dataset (right panel) — useful sanity check.
- **Gersbach HTv2 + Engreitz: modest downward bias on null classes** (~-0.02). Could be a calibration / normalization artifact worth flagging.

## 5. Doublet rate (clean test, both runs use sceptre)

Verified from `pipeline_info/params_*.json`: scrublet OFF and ON runs both have `GUIDE_ASSIGNMENT_method = sceptre`, only `ENABLE_SCRUBLET` differs. Cell-count drop is a real scrublet effect.

In [ ]:
scr = pd.read_csv(RESULTS / 'scrublet_compare' / 'cells_filtered.tsv', sep='\t')
scr = scr.set_index('dataset').reindex([d for d in DATASET_ORDER if d in scr['dataset'].values if d != ''] if 'dataset' in scr.columns else scr.index)
# Re-load preserving the index
scr = pd.read_csv(RESULTS / 'scrublet_compare' / 'cells_filtered.tsv', sep='\t', index_col=0)
scr = scr.reindex([d for d in DATASET_ORDER if d in scr.index])

fig, ax = plt.subplots(figsize=(9, 5))
ds_in = list(scr.index)
pct = scr['pct_dropped_on_vs_off'].values
colors = [DATASET_COLORS[d] for d in ds_in]
ax.bar(np.arange(len(ds_in)), pct, color=colors, alpha=0.9, edgecolor='white')
for xi, v in zip(np.arange(len(ds_in)), pct):
    ax.text(xi, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_xticks(np.arange(len(ds_in)))
ax.set_xticklabels([SHORT.get(d, d) for d in ds_in], fontsize=10)
ax.set_ylabel('% cells dropped (scrublet ON vs OFF)')
ax.set_title('Scrublet doublet rate per dataset', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '05_scrublet_drop.pdf', bbox_inches='tight')
plt.show()

print('\nCell counts and drop:')
scr_show = scr.copy()
scr_show.index = [SHORT.get(d, d) for d in scr_show.index]
display(scr_show.round(1))

**What stands out:**
- **Engreitz and Gersbach HTv2 are the doublet-heavy datasets** (15-18% of cells flagged). These two also showed the highest sensitivity to filter floor in section 3.
- **Gersbach GEM-Xv3 is the cleanest** (1% doublet rate).
- AUROC drops 0.005-0.018 with scrublet on (small in absolute terms) — the dominant scrublet effect is cell removal, not metric change.
- ⚠ For Hon, the no-scrublet cleanser run vs no-scrublet sceptre run differ in n_cells by 2.24× (56K vs 126K) — that's a real cleanser-vs-sceptre method difference, not a configuration mismatch.

## 6. Multiplet pile-up across filter floors

Computed from h5mu's `mod['guide'].layers['guide_assignment']` per cell.

In [ ]:
br = pd.read_csv(RESULTS / 'per_dataset_summary' / 'guide_assignment_breakdown.tsv', sep='\t')
br['short_name'] = br['dataset'].map(SHORT)

# >3 guides fraction across configs
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ds_in = [d for d in DATASET_ORDER if d in br['dataset'].values]

# Panel 1: across configs
for ds in ds_in:
    sub = br[br['dataset'] == ds].set_index('config_label').reindex(FILTER_CONFIG_ORDER)
    axes[0].plot(FILTER_CONFIG_ORDER, sub['frac_cells_more_than_3_guides'].values * 100,
                 'o-', color=DATASET_COLORS[ds], label=SHORT.get(ds, ds), linewidth=2, markersize=7)
axes[0].set_ylabel('% cells with >3 guides')
axes[0].set_title('Multiplet pile-up across filter floors', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=8, loc='upper left')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].tick_params(axis='x', rotation=20)

# Panel 2: 1/2/3/>3 stacked at best AUROC config
best_cfg = best.set_index('dataset')['config_label']
stack_data = []
for ds in ds_in:
    cfg = best_cfg.get(ds)
    sub = br[(br['dataset'] == ds) & (br['config_label'] == cfg)]
    if len(sub):
        r = sub.iloc[0]
        stack_data.append({
            'dataset': SHORT.get(ds, ds),
            'config': cfg,
            '1': r['frac_cells_exactly_1_guide'],
            '2': r['frac_cells_exactly_2_guides'],
            '3': r['frac_cells_exactly_3_guides'],
            '>3': r['frac_cells_more_than_3_guides'],
            '0': r['frac_cells_0_guides'],
        })
stack_df = pd.DataFrame(stack_data).set_index('dataset')
stack_df[['1', '2', '3', '>3', '0']].plot(kind='bar', stacked=True, ax=axes[1],
    color=['#2ca02c', '#ff7f0e', '#d62728', '#8c564b', '#7f7f7f'], width=0.7, alpha=0.9)
axes[1].set_ylabel('Fraction of cells')
axes[1].set_title('Guide assignment breakdown at each best config', fontsize=11, fontweight='bold')
axes[1].set_xticklabels(stack_df.index, rotation=20, ha='right', fontsize=9)
axes[1].legend(title='Guides/cell', fontsize=8, loc='center left', bbox_to_anchor=(1.02, 0.5))
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUT / '06_multiplet_breakdown.pdf', bbox_inches='tight')
plt.show()

print('\nGuide assignment breakdown at best config:')
display(stack_df.round(3))

**What stands out:**
- **Gersbach HTv2 at 2000 UMI: 25% of cells have >3 guides** — at the most restrictive UMI floor, the surviving cells are heavily multiplet-loaded. Real-world implication: stringent filtering on this dataset selects for high-MOI cells.
- **Hon has the highest baseline multiplet rate** (5-7% across all configs) — consistent with a higher MOI design intent (HTO chemistry).
- **Engreitz is the cleanest** (0.2-0.7% multiplet rate) but also has the lowest fraction of cells with any guide (0.18 at best config).
- **Gersbach pair at 200 UMI: most cells have ZERO guides** (gray = 87% / 75% with no guide) — confirms the 200-UMI cell explosion is largely empty barcodes.

## 7. Cross-dataset reproducibility per filter config

Spearman correlation of per-guide log2FC between dataset pairs at each filter config — does the same TF give the same effect on a different scRNA chemistry?

In [ ]:
corr = pd.read_csv(RESULTS / 'cross_parameter' / 'cross_dataset_log2fc_correlations.tsv', sep='\t')
config_present = [c for c in FILTER_CONFIG_ORDER if c in corr['run_label'].values]

# Box plot per config
fig, ax = plt.subplots(figsize=(9, 5))
data = [corr[corr['run_label'] == c]['r'].values for c in config_present]
bp = ax.boxplot(data, labels=config_present, patch_artist=True, widths=0.6)
for patch in bp['boxes']:
    patch.set_facecolor('#cccccc')
    patch.set_alpha(0.7)
for c, vals in zip(config_present, data):
    x = config_present.index(c) + 1
    ax.scatter([x] * len(vals), vals, color='steelblue', alpha=0.7, s=40, zorder=3)
ax.set_ylabel('Spearman ρ (per-guide log2FC across dataset pairs)')
ax.set_title('Cross-dataset reproducibility per filter config', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '07_cross_dataset_correlation.pdf', bbox_inches='tight')
plt.show()

# Per-pair table at 800 UMI (the most-complete config)
print('\nPairwise correlations at 800 UMI:')
tbl_800 = corr[corr['run_label'] == '800 UMI'][['pair', 'r', 'n']].set_index('pair').round(3)
display(tbl_800)

**What stands out:**
- **Flat UMI floors (500/800) give cross-dataset ρ of 0.66-0.86** — the bulk of TF effects reproduce across labs.
- **Knee2 crashes the distribution** — driven by Engreitz, suspected lane-splitting artifact.
- **2000 UMI tightens the high end** but loses many guides (only most-detected ones survive). Possibly worth a closer look — is high reproducibility there because we've selected for the most signal-loaded guides?
- **The headline manuscript finding**: a flat 500-800 UMI floor produces stable cross-lab correlation; the automatic Knee inflection is unstable for at least one lab.

## 8. Summary: findings, open questions, asks

### Findings

1. **Filter choice matters most for cross-lab reproducibility.** Flat 500-800 UMI floor: ρ = 0.66-0.86 across pairs. Knee2 inflection: crashes on Engreitz to 0.44.
2. **AUROC alone is not the right metric** for picking a config — need to balance against `frac_cells_with_guide × n_cells` or `ratio_obs_to_expected` to avoid false high-AUROC scores on tiny effective subsets (Gersbach pair at 200 UMI).
3. **The cleanest "sweet spot" per dataset:** Hon=500, Huangfu=Knee2 (or 500), Gersbach pair=800 (200 admits empties), Engreitz=500.
4. **Doublet rates differ 18× across technologies**: Gersbach GEM-Xv3 1.1% → Gersbach HTv2 18.4%. Scrublet drops AUROC by only 0.005-0.018 — cell removal is the dominant effect.
5. **Hon is severely under-expected** at every filter (~30% of expected 210K cells). Likely HTO demultiplex overhead.
6. **The benchmark guide library has 5 classes**: 30 NT, 54 OR negative controls, 8 PC, 324 TF. NT and OR null behave similarly across datasets (good sanity check).

### Open questions

1. **Is the Engreitz Knee2 crash real or a lane-splitting artifact?** Hardcoded as a caveat in `plot_params_sweep.py:43`. Need to confirm or rule out before pitching the finding as biology.
2. **Why does Hon recover only ~30% of expected cells** regardless of filter? HTO demux overhead seems likely but we haven't quantified.
3. **The modest negative-class log2FC bias on Gersbach HTv2 + Engreitz** — calibration artifact or real?
4. **Cleanser vs sceptre** — for Hon, sceptre admits 2.24× more cells than CLEANSER at 800 UMI. Is that consistent across other datasets / configs?

### Asks for Lucas (when pipeline bandwidth allows)

1. Retry **knee1** — currently 1/5 (Engreitz only completed). Most informative missing config.
2. Retry **multimap_kallisto** — stalled at prepare/.
3. **Gersbach pair at 500 UMI** — currently 3/5 only (Hon, Huangfu, Engreitz).
4. **Gersbach HTv2 at Knee2** — currently 4/5.
5. Stricter naming convention for sweep folder names (`Benchmark_scrublet_cleanser_*` actually used sceptre — caused real misinterpretation today).

### Adam's local follow-ups (no pipeline rerun needed)

1. Mito threshold sweep (10/15/20/25) on existing h5mu — half-day script.
2. SoupX / DecontX on Hon benchmark — trans DEG calibration diagnostic.
3. Sceptre vs Perturbo inference comparison from the same h5mu.

---

**Source of all data:**
- Per-axis outputs: [results/cross_tech_comparison/](../results/cross_tech_comparison/) (cross_parameter, cells_vs_expected, per_dataset_summary, scrublet_compare)
- Narrative status: [scratch/2026_04_28_manuscript_brainstorm/benchmark_sweep_status.md](../../../scratch/2026_04_28_manuscript_brainstorm/benchmark_sweep_status.md)
- Run inventory: [datasets/technology-benchmark_WTC11_TF-Perturb-seq/bin/1_get_data/gcp_inference_mudata_manifest.csv](bin/1_get_data/gcp_inference_mudata_manifest.csv)
- This notebook's plots: [results/initial_comparison/](../results/initial_comparison/)